# 31 — Calibration predictions (Kaggle, 2×T4)

Predictions of the Stage B model (`stage_b-full-s42-r1`) on the three calibration sets of `jevlite.calibration` (todo, 2026-10-30). The fitting itself runs on the laptop (`scripts/calibrate.py`, no GPU).

| Set | Records | Layer | Max length | GPU |
|---|---|---|---|---|
| `ragbench_calib`: train + validation of RAGBench covidqa, cuad, delucionqa, emanual, minus passages shared with the RAGBench test, ≤ 1000 per subset (`docs/judge_bench/calib/ragbench_calib_ids.txt`) | 2,972 | judge (Bool threshold) | 8192, as the judge benchmarks | 0 |
| `stage_a_calib`: `calib` of the Stage A mix | 19,921 | model (temperatures) | as saved (1024) | 1 |
| `stage_b_calib`: `calib` of the Stage B mix | 10,182 | model (temperatures) | as saved (1024) | 1 |

- **Inputs:** Add Input → Datasets → `mrgrin548/jevlite-mix` (Stage A `calib`), and Add Input → Notebook Output → the `06_build_stage_b` version that printed `MIX OK` (Stage B `calib`).
- **Settings:** Accelerator → GPU T4 x2, Internet → On, secret `HF_TOKEN` attached (the model is in the private repo `Nefelim584/jevlite-runs`).
- **Time:** ~25–30 min (GPU 0: CUAD contracts are long, ~1 s each; GPU 1: ~20 min).
- **Smoke run first:** all cells with `LIMIT = 40` (40 records per set), then `LIMIT = None` and **Save Version → Save & Run All (Commit)**.
- **Hand-off** (on the laptop): last cell.

In [ ]:
# Environment: code from GitHub at a fixed commit, kept outside /kaggle/working so it is not saved as output.
COMMIT = "Judge_bench"   # branch or commit hash; a hash makes the run reproducible

!rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
!cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
!pip install -q loguru
!python --version
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Parameters
LIMIT = 40                        # smoke run: records per set; None = full sets
MODEL = "hub:Nefelim584/jevlite-runs/stage_b-full-s42-r1/final"
REPO = "/tmp/JEV_Judge"
WORK = "/tmp/calib"               # converted data and the model: not saved
OUT = "/kaggle/working"           # notebook output: predictions, logs, the Stage B calib records

# One queue per GPU; the runs of a queue go one after another. name: predict_baseline.py arguments
QUEUES = {
    0: {"ragbench_calib": "--baseline jevlite --model {model} --max-len 8192 --fp16"},
    1: {"stage_a_calib": "--baseline jevlite --model {model} --fp16",
        "stage_b_calib": "--baseline jevlite --model {model} --fp16"},
}

## 1. Model and data

`ragbench_calib` is converted here (deterministic ids) and cut to the frozen id list. The two mix `calib` files are found among the inputs; the Stage B one is copied to the output, since the laptop has no copy of the Stage B mix.

In [ ]:
import glob, json, os, random, shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
owner, repo, sub = MODEL.removeprefix("hub:").split("/", 2)   # same as jevlite.model.resolve_checkpoint
MODEL_DIR = Path(snapshot_download(f"{owner}/{repo}", allow_patterns=[f"{sub}/*"], token=os.environ.get("HF_TOKEN"),
                                   local_dir=f"{WORK}/model")) / sub
print("model:", MODEL_DIR, sorted(os.listdir(MODEL_DIR)))
assert (MODEL_DIR / "heads.pt").exists(), "not a save_final folder"

DATA = Path(WORK) / "data"
subprocess.run([sys.executable, f"{REPO}/scripts/convert_public.py", "--datasets", "ragbench_calib",
                "--out", str(DATA), "--raw-dir", f"{WORK}/raw"], check=True, cwd=REPO)
keep = set(Path(f"{REPO}/docs/judge_bench/calib/ragbench_calib_ids.txt").read_text().split())
rows = [line for line in (DATA / "ragbench_calib.jsonl").open() if json.loads(line)["id"] in keep]
assert len(rows) == len(keep), f"{len(rows)} of {len(keep)} frozen ids found: the conversion changed"
(DATA / "ragbench_calib.cut.jsonl").write_text("".join(rows))

def find(pattern: str) -> str:
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"no {pattern} among the inputs (see the Inputs note at the top)"
    return hits[0]

SETS = {"ragbench_calib": str(DATA / "ragbench_calib.cut.jsonl"),
        "stage_a_calib": find("stage_a/calib.jsonl"),
        "stage_b_calib": find("stage_b/calib.jsonl")}
Path(OUT, "data").mkdir(exist_ok=True)
shutil.copy(SETS["stage_b_calib"], Path(OUT, "data", "stage_b_calib.jsonl"))

In [ ]:
# Input files: the full sets, or a fixed random sample of each for the smoke run.
def sample(name: str, path: str, n: int) -> str:
    rows = open(path).readlines()
    rows = random.Random(0).sample(rows, min(n, len(rows)))
    out = DATA / "smoke" / f"{name}.jsonl"
    out.parent.mkdir(exist_ok=True)
    out.write_text("".join(rows))
    return str(out)

FILES = {name: sample(name, path, LIMIT) if LIMIT else path for name, path in SETS.items()}
for name, f in FILES.items():
    print(name, f, sum(1 for _ in open(f)))

## 2. Predictions

Each GPU runs its queue in a background process; logs go to `OUT/logs/<run>.log`. `--resume` keeps finished predictions, so re-running this cell after an interruption continues where it stopped (in the same session).

In [ ]:
PREDS, LOGS = Path(OUT) / "preds", Path(OUT) / "logs"
PREDS.mkdir(exist_ok=True); LOGS.mkdir(exist_ok=True)

def queue_script(runs: dict) -> str:
    lines = ["set -x"]
    for name, args in runs.items():
        a = args.format(model=MODEL_DIR)
        lines.append(f"{sys.executable} {REPO}/scripts/predict_baseline.py {a} --data {FILES[name]} "
                     f"--out {PREDS}/{name}.jsonl --resume > {LOGS}/{name}.log 2>&1")
    return "\n".join(lines)

procs = {}
for gpu, runs in QUEUES.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs[gpu] = subprocess.Popen(["bash", "-c", queue_script(runs)], env=env, cwd=REPO)
    print(f"GPU {gpu}: {', '.join(runs)} (pid {procs[gpu].pid})")

In [ ]:
# Wait for both queues, printing the last progress line of every log each minute.
import time

def progress(name: str) -> str:
    log = LOGS / f"{name}.log"
    if not log.exists():
        return "waiting"
    text = log.read_text().replace("\r", "\n").strip().splitlines()
    return text[-1][:160] if text else "starting"

while any(p.poll() is None for p in procs.values()):
    print(time.strftime("%H:%M"), " | ".join(f"{n}: {progress(n)}" for runs in QUEUES.values() for n in runs), flush=True)
    time.sleep(60)
for gpu, p in procs.items():
    print(f"GPU {gpu}: exit code {p.returncode}")
for runs in QUEUES.values():
    for n in runs:
        print(f"== {n}:", progress(n))

## Hand-off (on the laptop, from the repo root)

```bash
kaggle kernels output <user>/<notebook-slug> -p data/calib   # preds/, logs/, data/stage_b_calib.jsonl
```

Then fit and apply (no GPU): the model layer on both mix `calib` sets, the judge layer on `ragbench_calib` (`data/calib/ragbench_calib.jsonl` is the same frozen cut, built locally):

```bash
uv run python scripts/calibrate.py fit \
  --data data/mix/stage_a/calib.jsonl data/calib/data/stage_b_calib.jsonl \
  --pred data/calib/preds/stage_a_calib.jsonl data/calib/preds/stage_b_calib.jsonl \
  --judge-data data/calib/ragbench_calib.jsonl --judge-pred data/calib/preds/ragbench_calib.jsonl \
  --out data/calib/calibration.json
```